<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>


# Crypto Foundations
## Chapter 8 – Secure Transport and Storage

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH<br>
The Crypto Engineer | AI-Powered by GPT 5.1.


The first sketch builds a toy envelope-encryption record in memory. You separate the data-encryption key (DEK) from the key-encryption key (KEK) and store a structured record with nonce, ciphertext, wrapped key, and metadata—mirroring the tuple you would persist in a real system.

In [1]:
import os  # random bytes for keys and nonces

def xor_stream(data: bytes, key: bytes) -> bytes:
    """Toy XOR-based "encryption" to illustrate structure, not security."""
    return bytes(d ^ k for d, k in zip(data, key))

dek = os.urandom(32)  # data-encryption key (DEK)
kek = os.urandom(32)  # key-encryption key (KEK)
nonce = os.urandom(12)  # nonce that would pair with AEAD in real code

plaintext = b"example document contents"  # data to protect
stream = os.urandom(len(plaintext))  # toy keystream derived from DEK+nonce in reality

ciphertext = xor_stream(plaintext, stream)
wrapped_dek = xor_stream(dek, kek)  # protect DEK with KEK

record = {
    "nonce": nonce.hex(),
    "ciphertext": ciphertext.hex(),
    "wrapped_dek": wrapped_dek.hex(),
    "metadata": {"name": "example.txt", "version": 1},
}

print("Envelope record:", record)


Envelope record: {'nonce': 'ac2d08fb368de3a21458d46c', 'ciphertext': '531b63a8fff9c20fd7329961822523cc696db4f4190e564503', 'wrapped_dek': '7d4a0da78310259dcdf03a976272345cfc05c874eb6c48610c00c58ee1ad8c02', 'metadata': {'name': 'example.txt', 'version': 1}}


The final lab sketches a minimal TLS client using the Python standard library. It connects to a hostname, performs a TLS handshake with default certificate authorities, and then inspects the negotiated protocol version and peer certificate—highlighting client-side checks that support end-to-end security.

In [2]:
import ssl
import socket

hostname = "example.com"  # expected server name
port = 443

context = ssl.create_default_context()  # load system CAs and sane defaults

with socket.create_connection((hostname, port)) as sock:
    with context.wrap_socket(sock, server_hostname=hostname) as tls_sock:
        cert = tls_sock.getpeercert()  # server certificate as a dict
        print("TLS version:", tls_sock.version())
        print("Certificate subject:", cert.get("subject"))


TLS version: TLSv1.3
Certificate subject: ((('commonName', 'example.com'),),)


<img src="https://hilpisch.com/tce_logo.png" alt="The Crypto Engineer" width="25%" align="right" border="0"><br>
